In [ ]:
import polars as pl
import requests
from lets_plot import *
LetsPlot.setup_html(isolated_frame=True)

url = "https://github.com/byuidatascience/data4missing/raw/master/data-raw/flights_missing/flights_missing.json"
df = pl.read_json(requests.get(url).content)

In [ ]:
df.head()

In [ ]:

df.shape

In [ ]:
df.schema

In [ ]:

df.describe()
# The problem is num_of_delays_carrier is str "1500+" 
# and num_of_delays_late_aircraft has the number -999. 

airport_code,airport_name,month,year,num_of_flights_total,num_of_delays_carrier,num_of_delays_late_aircraft,num_of_delays_nas,num_of_delays_security,num_of_delays_weather,num_of_delays_total,minutes_delayed_carrier,minutes_delayed_late_aircraft,minutes_delayed_nas,minutes_delayed_security,minutes_delayed_weather,minutes_delayed_total
str,str,str,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""ATL""","""Atlanta, GA: Hartsfield-Jackso…","""January""",2005,35048,"""1500+""",-999,4598,10,448,8355,116423,104415,207467,297,36931,465533
"""DEN""","""Denver, CO: Denver Internation…","""January""",2005,12687,"""1041""",928,935,11,233,3153,53537,70301,36817,363,21779,182797
"""IAD""","""""","""January""",2005,12381,"""414""",1058,895,4,61,2430,null,70919,35660,208,4497,134881
"""ORD""","""Chicago, IL: Chicago O'Hare In…","""January""",2005,28194,"""1197""",2255,5415,5,306,9178,88691,160811,364382,151,24859,638894
"""SAN""","""San Diego, CA: San Diego Inter…","""January""",2005,7283,"""572""",680,638,7,56,1952,27436,38445,21127,218,4326,91552


In [31]:
(
    df
    .filter(~pl.col("num_of_delays_carrier").str.contains(r"^\d+$"))
    ["num_of_delays_carrier"]
    .value_counts()
)

num_of_delays_carrier,count
str,u32
"""1500+""",73


In [32]:
df.filter(pl.col("num_of_delays_late_aircraft") == -999).height

40

In [ ]:
(
    df
    .filter(~pl.col("num_of_delays_carrier").str.contains(r"^\d+$"))
    ["num_of_delays_carrier"]
    .value_counts()
)

num_of_delays_carrier,count
str,u32
"""1500+""",73


In [ ]:
# "1500+" means a min of 1500. I will use it as a minimal value
# -999 means missing. I will consider it as null
# The column num_of_delays_carrier had the value “1500+” in [ ] rows, which made the whole column text. Since “1500+” means at least 1500, I replaced it with 1500 (this slightly underestimates carrier delays). The column num_of_delays_late_aircraft had -999 in [ ] rows, which means missing data. I changed it to null and filled it with the column mean, so those months are not counted as zero delays.

df_clean = (
    df
    .with_columns(
        pl.col("num_of_delays_carrier")
          .replace("1500+", "1500")
          .cast(pl.Int64),
        pl.col("num_of_delays_late_aircraft")
          .replace(-999, None),
    )
    .with_columns(
        pl.col("num_of_delays_late_aircraft")
          .fill_null(pl.col("num_of_delays_late_aircraft").mean().round(0))
          .cast(pl.Int64),
    )
)

In [34]:
wide = df_clean.select(
(pl.col("num_of_delays_carrier").sum() / pl.col("num_of_flights_total").sum()).alias("Carrier"),
(pl.col("num_of_delays_security").sum() / pl.col("num_of_flights_total").sum()).alias("Security"),
(pl.col("num_of_delays_late_aircraft").sum() / pl.col("num_of_flights_total").sum()).alias("Late Aircraft"),
)
long_df = (
    wide
    .unpivot(
        on=["Carrier", "Security", "Late Aircraft"],
        variable_name="delay_type",
        value_name="proportion",)
    .with_columns((pl.col("proportion") * 100).round(2).alias("percent"))
    .sort("proportion", descending=True)
    
)

In [24]:
long_df.select("delay_type", "percent")

delay_type,percent
str,f64
"""Late Aircraft""",6.68
"""Carrier""",4.55
"""Security""",0.03


In [ ]:
(
    ggplot(long_df, aes(x="delay_type", y="percent"))
    + geom_bar(stat="identity")
    + scale_x_discrete(limits=long_df["delay_type"].to_list())
    + labs(
        title="Late Aircraft had the most was the biggest reason for flights delayed, followed by carrier, and the last reason is security.",
        x="Delay type",
        y="% of flights delayed",
    )
)

In [ ]:
by_airport = (
    df_clean
    .group_by("airport_code")
    .agg(
        (pl.col("num_of_delays_carrier").sum() / pl.col("num_of_flights_total").sum()).alias("Carrier"),
        (pl.col("num_of_delays_security").sum() / pl.col("num_of_flights_total").sum()).alias("Security"),
        (pl.col("num_of_delays_late_aircraft").sum() / pl.col("num_of_flights_total").sum()).alias("Late Aircraft"),
    )
    .sort("airport_code")
)

long_airport = (
    by_airport
    .unpivot(
        index="airport_code",
        on=["Carrier", "Security", "Late Aircraft"],
        variable_name="delay_type",
        value_name="proportion",
    )
    .with_columns((pl.col("proportion") * 100).round(2).alias("percent"))
)

(
    ggplot(long_airport, aes(x="airport_code", y="percent", fill="delay_type"))
    + geom_bar(stat="identity", position="dodge")
    + labs(
        title="Late Aircraft is by far the biggest problem for  all airports.",
        x="Airport",
        y="% of flights delayed",
        fill="Delay type",
    )
)

In [27]:
# long -> wide: airports as rows, delay types as columns (in %)
long_airport.pivot(
    on="delay_type",
    index="airport_code",
    values="percent",
)

airport_code,Carrier,Security,Late Aircraft
str,f64,f64,f64
"""ATL""",3.91,0.02,5.32
"""DEN""",4.85,0.04,7.16
"""IAD""",5.61,0.03,7.61
"""ORD""",4.07,0.02,7.51
"""SAN""",6.22,0.05,7.69
"""SFO""",5.3,0.04,7.61
"""SLC""",4.63,0.06,5.66
